# Air Quality ETL Pipeline

This notebook walks through the Madagascar air quality ETL pipeline: from Visual Crossing API to a star schema in CSV and PostgreSQL.

## Pipeline Overview

```
Visual Crossing API
       |
AirQualityExtractor
  - historical (last 5 days)
  - forecast (next 5 days)
  - today_hourly (24 hours)
       |
DataFrameCleaner + DataValidator + DataAuditor
       |
Processed: data/clean/daily_air_quality_combined.csv
       |
AirQualityTransformer -> dim_city, dim_date, fact_aqi, fact_aqi_today
       |
CsvLoader (data/star_schema/) + PostgresLoader (PostgreSQL)
```

## Configuration

`config/settings.py` centralises all paths, env vars, and validation. On startup, `Settings.validate()` checks every required variable before any API call. This fails fast instead of 5 minutes into extraction.

In [ ]:
from config.settings import Settings
from config.logging import setup_logging

setup_logging()
print(f"API Base URL: {Settings.VISUAL_CROSSING_BASE_URL}")
print(f"Raw data: {Settings.RAW_DIR}")
print(f"Clean data: {Settings.CLEAN_DIR}")
print(f"Star schema: {Settings.STAR_SCHEMA_DIR}")
print(f"Fact path: {Settings.AQI_FACT_PATH}")

In [ ]:
try:
    Settings.validate()
    Settings.ensure_directories()
    print("All settings valid. Directories ready.")
except ValueError as e:
    print(f"Validation failed: {e}")

## DataValidator - Range Checking

Checks every numeric column against known valid ranges. If the API returns `pm2.5: 9999` (sensor error), the validator flags it before the bad value reaches the star schema.

In [ ]:
from src.transform.quality.data_validator import DataValidator

for col, (lo, hi) in DataValidator.RANGES.items():
    print(f"  {col:20s}  [{lo:>6}, {hi:>6}]")
print(f"\nTotal metrics tracked: {len(DataValidator.RANGES)}")

In [ ]:
import pandas as pd
import logging

# Good data - all within range
good = pd.DataFrame({"pm2.5": [10, 25, 35], "pm10": [20, 40, 60]})
print("=== Valid data ===")
DataValidator.validate(good, "Good Example")

print()

# Bad data - sensor spike (999 is out of range)
bad = pd.DataFrame({"pm2.5": [10, 999, 35], "o3": [30, 40, 9999]})
print("=== Corrupted data (should warn) ===")
DataValidator.validate(bad, "Bad Example")

## DataFrameCleaner - Data Sanitisation

Six composable methods that fix common data problems:
- Empty strings (`""`, `"  "`) are converted to NaN, then filled
- Duplicate rows are removed
- Nulls in numeric columns are filled with 0
- Nulls in categorical columns are filled with "unknown"
- Columns that are entirely null are dropped

In [ ]:
from src.transform.quality.dataframe_cleaner import DataFrameCleaner

# Simulate messy raw data
messy = pd.DataFrame({
    "pm2.5": [10.0, None, 20.0, None],
    "pm10": [None, 15.0, 25.0, None],
    "city_name": ["Tana", "", "  ", "Fianar"],
    "useless": [None, None, None, None],
})

print("=== BEFORE cleaning ===")
print(messy)
print(f"\nNull counts:\n{messy.isnull().sum()}")

clean = DataFrameCleaner.clean_air_quality_data(messy)

print("\n=== AFTER cleaning ===")
print(clean)
print(f"\nNull counts:\n{clean.isnull().sum()}")

# Notice:
# - "" and "  " became "unknown"
# - pm2.5 and pm10 nulls became 0
# - useless column was dropped entirely

## DataAuditor - Quality Observability

Runs a full health check on any DataFrame: shape, nulls, duplicates, numeric stats, categorical distribution. The validator checks values are in range; the auditor checks structure. If one city starts returning all nulls, the auditor catches it immediately.

In [ ]:
from src.transform.quality.data_auditor import DataAuditor

auditor = DataAuditor()

sample = pd.DataFrame({
    "pm2.5": [10.0, None, 35.0, 20.0, 15.0],
    "pm10": [20.0, 30.0, None, 40.0, 25.0],
    "city_name": ["Tana", "Tana", "Tana", "Fianar", "Fianar"],
})

report = auditor.audit_dataframe(sample, "Demo Data")

print(f"Rows: {report['basic_info']['row_count']}")
print(f"Cols: {report['basic_info']['column_count']}")
print(f"Total nulls: {report['null_analysis']['total_nulls']}")
print(f"Duplicates: {report['duplicate_analysis']['duplicate_count']}")
print(f"Numeric columns tracked: {list(report['numeric_statistics'].keys())}")

## Data Flow

### 1. Extract
Each city goes through three extractions:
- Historical: last 5 days, daily snapshot, never changes
- Forecast: next 5 days, changes every API call
- Hourly: today's 24 hours, high volume

Three folders keep them separate because they have different columns and update frequencies.

In [ ]:
print("Extraction produces 3 DataFrames per city:")
print("  historical  -> data/raw/historical/{city}_historical.csv")
print("  forecast    -> data/raw/forecast/{city}_forecast.csv")
print("  today_hrly  -> data/raw/today_hourly/{city}_today_hourly.csv")

### 2. Clean
Every raw DataFrame passes through `DataFrameCleaner.clean_air_quality_data()`:
1. `normalize_empty_strings()` - convert "" and "  " to NaN
2. `remove_duplicates()` - drop exact row copies
3. `fill_numeric_nulls()` - NaN to 0
4. `fill_categorical_nulls()` - NaN to 'unknown'

### 3. Validate + Audit
Before anything enters the star schema, both auditor and validator run on every fact table. If they log warnings, the pipeline still completes but warnings appear in production logs for review.

### 4. Star Schema

```
dim_city              dim_date
--------              --------
city_key PK           date_key PK
city_name             full_date
region                year
latitude              month
longitude             day
population            day_of_week
                      quarter
     \\               //
      \\             //
      fact_aqi / fact_aqi_today
      -------------------------
      city_key FK -> dim_city
      date_key FK -> dim_date
      hour (only fact_aqi_today)
      pm2.5, pm10, o3, no2, so2, co
```

### 5. Load

Two destinations:
- **CSV**: `data/star_schema/{dim_date,dim_city,fact_aqi,fact_aqi_today}.csv`
- **PostgreSQL**: `air_quality` schema with 4 tables, foreign keys, unique constraints

Strategy is DELETE + append (not REPLACE). REPLACE drops and recreates the table, killing foreign keys and indexes. DELETE + append preserves schema and is idempotent.

## Failure Scenarios

| Scenario | How We Prevent It |
|----------|-------------------|
| API key invalid | `Settings.validate()` fails before any API call |
| API rate limit (429) | `@retry` retries 3 times with backoff |
| API returns `pm2.5: -5` | `DataValidator.RANGES[pm2.5] = (0, 500)` flags it |
| Empty string in city name | `normalize_empty_strings()` converts to NaN, then 'unknown' |
| Duplicate rows on re-run | `remove_duplicates()` + UNIQUE constraint on facts |
| New column from API | `DataAuditor` logs every column found vs expected |
| PostgreSQL down | CSV files are saved first; Postgres is the last step |
| Missing city CSV | `CityExtractor` raises FileNotFoundError immediately |

## Running the Pipeline

One-shot:
```bash
python main.py
```

Scheduled via Airflow (DAG: `air_quality_madagascar_etl`, runs daily at 06:00):
```
validate_settings -> extract per city -> prepare_dimensions -> transform_facts -> save_postgres
```

Output:
- `data/raw/{historical,forecast,today_hourly}/{city}_{type}.csv`
- `data/clean/daily_air_quality_combined.csv`
- `data/star_schema/{dim_date,dim_city,fact_aqi,fact_aqi_today}.csv`
- PostgreSQL `air_quality` schema with 4 tables, foreign keys, unique constraints